# 03 — Summary Memory

Instead of **dropping** old messages when the buffer gets full, we **compress** them.
An LLM reads the old messages and writes a short summary. The summary takes up far
fewer tokens than the original messages.

### The idea
```
[Old msg 1]
[Old msg 2]   ──── LLM summarises ────▶  "[Summary: User is Meera, 28, likes travel]"
[Old msg 3]

 kept ↓
[Recent msg 4]
[Recent msg 5]
[New question ]
```

### What you will learn
| Concept | Description |
|---------|-------------|
| Summary memory | LLM compresses old messages into a single paragraph |
| Hybrid buffer | Keep a summary of old + full text of recent messages |
| Token comparison | See how summary saves tokens vs full buffer |
| When to use each | Choosing the right memory type for the situation |

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install & Import

In [ ]:
# !pip install langchain-groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

llm = ChatGroq(model="llama3-8b-8192", temperature=0.3)
print("Ready!")

## 2. Build the Summary Memory Class

In [ ]:
class SummaryMemory:
    """
    Hybrid memory: keeps a running LLM summary of old messages
    + the full text of the most recent messages.

    When the recent buffer exceeds MAX_RECENT, the oldest half
    is summarised and merged into self.summary.
    """

    MAX_RECENT = 6   # how many recent messages to keep verbatim

    def __init__(self):
        self.summary = ""        # compressed history (older messages)
        self.recent  = []        # full-text recent messages

    # ── Add a new message ────────────────────────────────────────────────
    def add(self, role: str, content: str):
        self.recent.append({"role": role, "content": content})
        # If we've exceeded the limit, compress the oldest half
        if len(self.recent) > self.MAX_RECENT:
            self._compress()

    # ── Compress old messages into a summary ─────────────────────────────
    def _compress(self):
        half      = len(self.recent) // 2
        to_summarise = self.recent[:half]
        self.recent  = self.recent[half:]   # keep the newer half verbatim

        # Build the text to summarise
        convo_text = "\n".join(
            f"{m['role'].title()}: {m['content']}"
            for m in to_summarise
        )
        prompt = (
            "Summarise the following conversation in 2–3 sentences. "
            "Keep all names, facts, and preferences mentioned.\n\n"
            + convo_text
        )
        new_summary = llm.invoke([HumanMessage(content=prompt)]).content.strip()

        # Merge with any existing summary
        if self.summary:
            self.summary = f"{self.summary} Later: {new_summary}"
        else:
            self.summary = new_summary

        print(f"  [Summary Memory] Compressed {half} messages.")
        print(f"  [Summary] {self.summary[:100]}...\n")

    # ── Build the context string to send to the LLM ──────────────────────
    def get_context(self) -> str:
        parts = []
        if self.summary:
            parts.append(f"[Earlier conversation summary: {self.summary}]")
        for m in self.recent:
            label = "User" if m["role"] == "user" else "Assistant"
            parts.append(f"{label}: {m['content']}")
        return "\n".join(parts)

    def token_estimate(self) -> dict:
        """Rough token count (1 token ≈ 4 characters)."""        summary_tokens = len(self.summary) // 4
        recent_tokens  = sum(len(m["content"]) // 4 for m in self.recent)
        return {
            "summary_tokens": summary_tokens,
            "recent_tokens":  recent_tokens,
            "total_tokens":   summary_tokens + recent_tokens
        }

print("SummaryMemory class defined!")

## 3. Build the Chatbot Around It

In [ ]:
SYSTEM_PROMPT = (
    "You are a friendly personal assistant. "
    "Use the conversation context below to give personalised, helpful replies."
)

mem = SummaryMemory()

def chat(user_input):
    # Get current context (summary + recent messages)
    context = mem.get_context()

    # Build the full prompt
    system_content = SYSTEM_PROMPT
    if context:
        system_content += f"\n\nConversation so far:\n{context}"

    messages = [
        SystemMessage(content=system_content),
        HumanMessage(content=user_input),
    ]

    response = llm.invoke(messages)
    reply    = response.content.strip()

    # Update memory
    mem.add("user",      user_input)
    mem.add("assistant", reply)

    print(f"You : {user_input}")
    print(f"Bot : {reply}\n")
    return reply

print("Chatbot ready! (SummaryMemory active)")

## 4. Have a Long Conversation

In [ ]:
# Round 1 — introduce yourself
chat("Hi! I'm Meera. I'm 28 years old.")
chat("I'm a product manager at a fintech startup in Pune.")
chat("I love travelling, especially solo trips to the mountains.")

In [ ]:
# Round 2 — more details
chat("I'm currently planning a trip to Ladakh in October.")
chat("I need help with packing because it'll be very cold there.")
chat("I also want some recommendations for things to see in Leh.")

In [ ]:
# Round 3 — test if memory survived the compression
# By this point the first messages should have been summarised
chat("What do you remember about me?")
chat("Can you remind me where I work?")
chat("What trip am I planning?")

## 5. Check Memory State

In [ ]:
print("=" * 50)
print("CURRENT MEMORY STATE")
print("=" * 50)

print(f"\nSummary (compressed older messages):")
if mem.summary:
    print(f"  {mem.summary}")
else:
    print("  (no summary yet — not enough messages)")

print(f"\nRecent messages ({len(mem.recent)} verbatim):")
for i, m in enumerate(mem.recent):
    preview = m['content'][:80] + "..." if len(m['content']) > 80 else m['content']
    print(f"  [{i+1}] {m['role']:9s}: {preview}")

print("\nToken estimate:")
t = mem.token_estimate()
for key, val in t.items():
    print(f"  {key:20s}: ~{val}")

## 6. Compare Token Usage — Buffer vs Summary

In [ ]:
# Simulate the same conversation kept as a FULL buffer
# and count how many tokens it would use

all_messages_text = []

# Replay all messages from memory
for m in mem.recent:
    all_messages_text.append(m["content"])

# A full buffer would have held every single message ever sent
# Let's estimate: 12 turns x avg 40 words x 1.3 tokens/word ≈ 624 tokens
full_buffer_estimate = 12 * 40  # rough

print("Token usage comparison for a 12-turn conversation:")
print(f"  Full buffer (all messages): ~{full_buffer_estimate} tokens")
print(f"  Summary memory            : ~{mem.token_estimate()['total_tokens']} tokens")
savings = full_buffer_estimate - mem.token_estimate()["total_tokens"]
pct     = (savings / full_buffer_estimate * 100) if full_buffer_estimate > 0 else 0
print(f"  Savings                   : ~{savings} tokens  ({pct:.0f}% reduction)")
print()
print("The longer the conversation, the bigger the saving!")

## 7. Which Memory Type Should You Use?

| Situation | Best Memory Type |
|-----------|----------------|
| Short Q&A session (< 10 turns) | Buffer Memory |
| Long conversation that must remember everything | Summary Memory |
| You only care about the last few turns | Window Memory (Notebook 02) |
| Conversation must survive app restart | Persistent Memory (Notebook 05) |

**Summary memory is the sweet spot** for most production chatbots — it keeps
the gist of every conversation without burning tokens on old messages.

**Next notebook →** LangGraph's built-in memory via `MemorySaver` checkpointers.